# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
import os
REPO_URL = "https://github.com/tam253211-a11y/K4-Track-4-Day10-DangHuuTam-2A202602940-VisionLanguageModel.git"
REPO_DIR = "/content/K4-Track-4-Day10-DangHuuTam-2A202602940-VisionLanguageModel"
if not os.path.exists("scripts/score.py"):          # chạy trên Colab: clone repo rồi cd vào
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
!nvidia-smi -L
!pip -q install -r requirements-colab.txt pytest

In [ ]:
# Kiểm tra công cụ của đề. Kỳ vọng: 68 passed, 1 skipped (test dữ liệu thật bị skip khi chưa tải dataset)
!python -m pytest -q

In [ ]:
# Google Drive để sao lưu adapter/projector sau mỗi lần chạy (Colab free có thể bị ngắt)
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/day10_vlm"
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
import json
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from scripts.labcommon import CLASSES, read_jsonl

DATA = Path("data")
splits = {s: read_jsonl(DATA / "splits" / f"{s}.jsonl") for s in ("train", "val", "test")}
meta = json.loads((DATA / "splits" / "meta.json").read_text(encoding="utf-8"))

# 1) dataset_version
print("dataset_version:", meta["dataset_version"])
assert meta["dataset_version"].startswith("d10-")

# 2) số ảnh / box từng split, so với bảng của đề
EXPECTED = {"train": (1422, 1074, 1750, 691), "val": (276, 66, 470, 103), "test": (311, 121, 299, 163)}
for s, rows in splits.items():
    c = Counter(o["label"] for r in rows for o in r["objects"])
    got = (len(rows), c["pothole"], c["crack"], c["manhole"])
    print(f"{s:5s} ảnh={got[0]:5d}  pothole={got[1]:5d}  crack={got[2]:5d}  manhole={got[3]:5d}  "
          f"sessions={len({r['session'] for r in rows})}  {'OK' if got == EXPECTED[s] else 'KHÁC ĐỀ!'}")

# 3) không session nào nằm ở hai split
sess = {s: {r["session"] for r in rows} for s, rows in splits.items()}
overlap = (sess["train"] & sess["val"]) | (sess["train"] & sess["test"]) | (sess["val"] & sess["test"])
print("session trùng giữa các split:", overlap or "không có")
assert not overlap

# 4) vẽ 6 ảnh train có box (YOLO chuẩn hoá [0,1] → pixel)
COLORS = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}

def draw_gt(rec):
    img = Image.open(DATA / rec["image_path"]).convert("RGB")
    d = ImageDraw.Draw(img)
    W, H = img.size
    for o in rec["objects"]:
        xc, yc, w, h = o["yolo"]
        box = [(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H]
        d.rectangle(box, outline=COLORS[o["label"]], width=3)
        d.text((box[0] + 2, box[1] + 2), o["label"], fill=COLORS[o["label"]])
    return img

samples = [r for r in splits["train"] if r["objects"]][::200][:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, rec in zip(axes.flat, samples):
    ax.imshow(draw_gt(rec)); ax.set_title(f"{rec['session']} · {len(rec['objects'])} box", fontsize=9); ax.axis("off")
plt.tight_layout()
Path("report/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("report/figures/data_samples.png", dpi=80, bbox_inches="tight")
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# YOUR CODE HERE: viết yolo_to_box() và objects_to_answer() (GUIDE bước 2)

## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9